# 03 · GPU preprocessing — agnostic masks, DensePose, worn-pair garments (T4, run once)
Run after notebook 02 has exported `DesiFit-VTON-v1`. Uses CatVTON's bundled **AutoMasker** (SCHP + DensePose), so no detectron2 install.

> Runtime → Change runtime type → **T4 GPU**. On Kaggle: GPU T4 + Internet on; change the Drive paths to `/kaggle/input/...` and `/kaggle/working/...`.

Writes into each split folder: `agnostic-mask/`, `image-densepose/`, `image-parse/`, and `cloth/` + `cloth-mask/` for worn pairs. Resumable.

## 0 · Setup
Mount Drive and get the code. Put **either** your GitHub URL in `REPO_URL`, **or** leave it empty to use `desi-fit-ai.zip` from the dataset folder on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
ROOT   = '/content/drive/MyDrive/AainaClosetDataset'      # scraped images live here
BUILD  = f'{ROOT}/_build'                                  # all intermediate CSVs + review pages
DATASET= '/content/drive/MyDrive/DesiFitAI/processed/DesiFit-VTON-v1'   # final exported dataset
REPO_URL = ''   # e.g. 'https://github.com/<you>/desi-fit-ai.git'  (private repo: use a token URL)
os.makedirs(BUILD, exist_ok=True)

if REPO_URL:
    !rm -rf /content/desi-fit-ai && git clone -q $REPO_URL /content/desi-fit-ai
else:
    !rm -rf /content/desi-fit-ai && unzip -q "$ROOT/desi-fit-ai.zip" -d /content/
%cd /content/desi-fit-ai
!git log -1 --oneline 2>/dev/null || ls

### CatVTON + fixes from the runbook
Strip the pinned torch/xformers, pin `huggingface_hub` (the old `diffusers` still imports `cached_download`). **Restart the session after this cell**, then re-run the setup cell above and continue below.

In [ ]:
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!git clone -q https://github.com/Zheng-Chong/CatVTON.git /content/CatVTON || true
!grep -v -E "^(torch|torchvision|xformers)" /content/CatVTON/requirements.txt > /content/req_colab.txt
!pip -q install -r /content/req_colab.txt
!pip -q install huggingface_hub==0.25.2 opencv-python-headless pandas
!pip freeze > /content/drive/MyDrive/DesiFitAI/working_env_masks.txt

### Smoke test: 12 pairs + QA sheet
Look at the red overlay. **For kameez the mask must reach the hem**, not stop at the hip. If it does not, open `data_tools/masks_gpu.py → extend_to_hem` and note the change in `docs/decisions.md`.

In [ ]:
import sys; sys.path.insert(0, '/content/desi-fit-ai')
from data_tools.masks_gpu import process, qa_sheet
qa = process(DATASET, ROOT, '/content/CatVTON', limit=12)
print(qa.head(12).to_string())
from IPython.display import Image as IImage
IImage(qa_sheet(DATASET, '/content/qa_sheet.png', n=12))

### Full run

In [ ]:
qa = process(DATASET, ROOT, '/content/CatVTON')
qa.status.str.split(':').str[0].value_counts()

### Statistics for the datasheet

In [ ]:
!python -m data_tools.stats --build "$BUILD" --export "$DATASET" > "$DATASET/dataset_stats.md"
from IPython.display import Markdown; Markdown(open(f'{DATASET}/dataset_stats.md').read())